In [ ]:
# =============================================================================
# Tik-Tok: Website Fingerprinting with Packet Timing Features
# Supports Closed-World and Open-World evaluation
# =============================================================================
 
import os
import numpy as np
import pandas as pd
import pickle
import random
from collections import defaultdict
 
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, BatchNormalization,
                                     Activation, Flatten, Dense, Dropout)
from tensorflow.keras.initializers import glorot_uniform
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle as sk_shuffle
 
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
 
# =============================================================================
# ★ USER CONFIGURATION — chỉnh ở đây
# =============================================================================
 
CLOSE_DATA_DIR = ""   # folder chứa closed-world data
OPEN_DATA_DIR  = ""    # folder chứa open-world (unmonitored) data
 
# Train/val/test split ratio cho closed-world (monitored)
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.10
# TEST_RATIO  = 0.20  (phần còn lại)
 
# Open-world: số lượng unmonitored sample dùng cho training và testing
UNMON_TRAIN_SIZE = 400   # số sample unmonitored dùng để train
UNMON_TEST_SIZE  = 10000   # số sample unmonitored dùng để test
 
# Số bins để normalize histogram features
BIN_SIZE = 20
 
# Training
BATCH_SIZE = 128
EPOCHS_CW  = 100   # closed-world
EPOCHS_OW  = 30    # open-world

In [ ]:
def parse_trace(filepath):
    """
    Read a trace file where each line contains <timestamp> <packet_size_with_direction>.
    Returns a list of [timestamp, direction] where direction = +1 (out) or -1 (in).
    """
    packets = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            try:
                ts  = float(parts[0])
                pkt = float(parts[1])
                packets.append([ts, 1 if pkt > 0 else -1])
            except ValueError:
                continue
    return packets


def extract_bursts(packets):
    """
    Group consecutive packets with the same direction into bursts.
    Returns a list of bursts, where each burst is a list of [timestamp, direction].
    """
    if not packets:
        return []
    bursts = []
    current_burst = [packets[0]]
    for pkt in packets[1:]:
        if pkt[1] == current_burst[-1][1]:
            current_burst.append(pkt)
        else:
            bursts.append(current_burst)
            current_burst = [pkt]
    bursts.append(current_burst)
    return bursts


def feat_MED(bursts):
    """Median packet timestamp within each burst."""
    return [np.median([p[0] for p in b]) for b in bursts]


def feat_Variance(bursts):
    """Timestamp variance within each burst."""
    return [np.var([p[0] for p in b]) for b in bursts]


def feat_Burst_Length(bursts):
    """Duration of each burst (last - first timestamp)."""
    return [b[-1][0] - b[0][0] for b in bursts]


def feat_IMD(bursts):
    """Inter-Median Delay: difference between consecutive burst MEDs."""
    meds = feat_MED(bursts)
    return [q - p for p, q in zip(meds[:-1], meds[1:])]


def feat_IBD_FF(bursts):
    """First-to-first inter-burst delay."""
    return np.diff([float(b[0][0]) for b in bursts]).tolist()


def feat_IBD_LF(bursts):
    """Last-to-first inter-burst delay (last packet of burst i to first of burst i)."""
    return [float(b[-1][0]) - float(b[0][0]) for b in bursts]


def feat_IBD_IFF(bursts):
    """First-to-first delay for incoming bursts only (direction == -1)."""
    incoming = [b for b in bursts if b[0][1] == -1]
    return np.diff([float(b[0][0]) for b in incoming]).tolist()


def feat_IBD_OFF(bursts):
    """First-to-first delay for outgoing bursts only (direction == +1)."""
    outgoing = [b for b in bursts if b[0][1] == 1]
    return np.diff([float(b[0][0]) for b in outgoing]).tolist()


FEATURE_EXTRACTORS = {
    "MED":          feat_MED,
    "Variance":     feat_Variance,
    "Burst_Length": feat_Burst_Length,
    "IMD":          feat_IMD,
    "IBD_FF":       feat_IBD_FF,
    "IBD_LF":       feat_IBD_LF,
    "IBD_IFF":      feat_IBD_IFF,
    "IBD_OFF":      feat_IBD_OFF,
}


def normalize_to_histogram(values, n_bins):
    """Convert a list of values into a normalized histogram of n_bins bins. Returns zeros if empty."""
    if len(values) == 0:
        return [0.0] * n_bins
    counts, _ = np.histogram(values, bins=n_bins)
    total = counts.sum()
    return (counts / total).tolist() if total > 0 else [0.0] * n_bins


def extract_feature_vector(filepath, n_bins=BIN_SIZE):
    """Read one trace file and return a feature vector (8 features × n_bins dimensions)."""
    packets = parse_trace(filepath)
    bursts  = extract_bursts(packets)

    vector = []
    for feat_name, extractor in FEATURE_EXTRACTORS.items():
        try:
            vals = extractor(bursts)
        except Exception:
            vals = []
        vector.extend(normalize_to_histogram(vals, n_bins))
    return vector  # length = 8 * n_bins

In [ ]:
# =============================================================================
# DATA LOADING
# =============================================================================
 
def load_dataset(data_dir, label_override=None, n_bins=BIN_SIZE):
    X, y = [], []
    label_dirs = sorted(os.listdir(data_dir))
 
    for label_idx, label_name in enumerate(label_dirs):
        label_path = os.path.join(data_dir, label_name)
        if not os.path.isdir(label_path):
            continue
 
        label = label_override if label_override is not None else label_idx
 
        files = sorted([f for f in os.listdir(label_path)
                        if os.path.isfile(os.path.join(label_path, f))])
        for fname in files:
            fpath = os.path.join(label_path, fname)
            try:
                vec = extract_feature_vector(fpath, n_bins)
                X.append(vec)
                y.append(label)
            except Exception as e:
                print(f"  [skip] {fpath}: {e}")
 
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)
 
 
def split_by_label(X, y, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, seed=42):
    rng = np.random.RandomState(seed)
    X_tr, y_tr = [], []
    X_vl, y_vl = [], []
    X_te, y_te = [], []
 
    for label in np.unique(y):
        idx = np.where(y == label)[0]
        rng.shuffle(idx)
        n = len(idx)
        n_tr = int(n * train_ratio)
        n_vl = int(n * val_ratio)
 
        X_tr.extend(X[idx[:n_tr]]);          y_tr.extend([label] * n_tr)
        X_vl.extend(X[idx[n_tr:n_tr+n_vl]]); y_vl.extend([label] * n_vl)
        X_te.extend(X[idx[n_tr+n_vl:]]);      y_te.extend([label] * (n - n_tr - n_vl))
 
    def to_np_shuffle(Xl, yl):
        Xa, ya = np.array(Xl, dtype=np.float32), np.array(yl, dtype=np.int32)
        Xa, ya = sk_shuffle(Xa, ya, random_state=seed)
        return Xa, ya
 
    return (*to_np_shuffle(X_tr, y_tr),
            *to_np_shuffle(X_vl, y_vl),
            *to_np_shuffle(X_te, y_te))

In [ ]:
# =============================================================================
# MODEL — DeepFingerprinting CNN (DFNet)
# =============================================================================
 
def build_model(num_classes, seq_length=160):
    model = Sequential([
        # Block 1 — ELU
        Conv1D(32, 8, strides=1, padding='same', input_shape=(seq_length, 1), name='b1_conv1'),
        BatchNormalization(),
        Activation('elu', name='b1_act1'),
        Conv1D(32, 8, strides=1, padding='same', name='b1_conv2'),
        BatchNormalization(),
        Activation('elu', name='b1_act2'),
        MaxPooling1D(pool_size=8, strides=4, padding='same', name='b1_pool'),
        Dropout(0.1, name='b1_drop'),
 
        # Block 2 — ReLU
        Conv1D(64, 8, strides=1, padding='same', name='b2_conv1'),
        BatchNormalization(),
        Activation('relu', name='b2_act1'),
        Conv1D(64, 8, strides=1, padding='same', name='b2_conv2'),
        BatchNormalization(),
        Activation('relu', name='b2_act2'),
        MaxPooling1D(pool_size=8, strides=4, padding='same', name='b2_pool'),
        Dropout(0.1, name='b2_drop'),
 
        # Block 3 — ReLU
        Conv1D(128, 8, strides=1, padding='same', name='b3_conv1'),
        BatchNormalization(),
        Activation('relu', name='b3_act1'),
        Conv1D(128, 8, strides=1, padding='same', name='b3_conv2'),
        BatchNormalization(),
        Activation('relu', name='b3_act2'),
        MaxPooling1D(pool_size=8, strides=4, padding='same', name='b3_pool'),
        Dropout(0.1, name='b3_drop'),
 
        # Block 4 — ReLU
        Conv1D(256, 8, strides=1, padding='same', name='b4_conv1'),
        BatchNormalization(),
        Activation('relu', name='b4_act1'),
        Conv1D(256, 8, strides=1, padding='same', name='b4_conv2'),
        BatchNormalization(),
        Activation('relu', name='b4_act2'),
        MaxPooling1D(pool_size=8, strides=4, padding='same', name='b4_pool'),
        Dropout(0.1, name='b4_drop'),
 
        Flatten(name='flatten'),
 
        Dense(512, kernel_initializer=glorot_uniform(seed=0), name='fc1'),
        BatchNormalization(),
        Activation('relu', name='fc1_act'),
        Dropout(0.7, name='fc1_drop'),
 
        Dense(512, kernel_initializer=glorot_uniform(seed=0), name='fc2'),
        BatchNormalization(),
        Activation('relu', name='fc2_act'),
        Dropout(0.5, name='fc2_drop'),
 
        Dense(num_classes, kernel_initializer=glorot_uniform(seed=0), name='fc_out'),
        Activation('softmax', name='softmax'),
    ])
 
    model.compile(
        loss='categorical_crossentropy',
        optimizer=Adamax(learning_rate=0.002, beta_1=0.9, beta_2=0.999, epsilon=1e-8),
        metrics=['accuracy']
    )
    return model
 
 

In [ ]:
# =============================================================================
# CLOSED-WORLD EVALUATION
# =============================================================================

from sklearn.metrics import classification_report
import numpy as np

def run_closed_world():
    print("=" * 60)
    print("CLOSED-WORLD EVALUATION")
    print("=" * 60)

    print("\n[1/3] Loading & extracting features from closed-world data...")
    X, y = load_dataset(CLOSE_DATA_DIR)
    print(f"  Total samples: {len(y)}, Classes: {len(np.unique(y))}")

    print("\n[2/3] Splitting data (70/10/20)...")
    X_tr, y_tr, X_vl, y_vl, X_te, y_te = split_by_label(X, y)
    print(f"  Train: {len(y_tr)} | Val: {len(y_vl)} | Test: {len(y_te)}")

    num_classes = len(np.unique(y))
    seq_length  = X_tr.shape[1]

    # Reshape cho Conv1D: (N, L, 1)
    X_tr_c = X_tr[:, :, np.newaxis]
    X_vl_c = X_vl[:, :, np.newaxis]
    X_te_c = X_te[:, :, np.newaxis]

    Y_tr = to_categorical(y_tr, num_classes)
    Y_vl = to_categorical(y_vl, num_classes)
    Y_te = to_categorical(y_te, num_classes)

    print(f"\n[3/3] Training DFNet ({num_classes} classes, seq_len={seq_length})...")
    model = build_model(num_classes, seq_length)

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )

    model.fit(
        X_tr_c, Y_tr,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS_CW,
        validation_data=(X_vl_c, Y_vl),
        callbacks=[early_stop],
        verbose=2
    )

    # -----------------------------------------------------------------
    # Evaluate
    # -----------------------------------------------------------------
    loss, acc = model.evaluate(X_te_c, Y_te, verbose=0)
    print(f"\n★ Closed-World Test Accuracy: {acc:.4f} (loss={loss:.4f})")

    # -----------------------------------------------------------------
    # Classification Report
    # -----------------------------------------------------------------
    y_prob = model.predict(X_te_c, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    print("\n" + "=" * 60)
    print("CLASSIFICATION REPORT")
    print("=" * 60)

    print(classification_report(
        y_te,
        y_pred,
        digits=4,
        zero_division=0
    ))

    return model, acc

In [ ]:
# =============================================================================
# OPEN-WORLD EVALUATION
# =============================================================================
def ow_evaluation(model, X_mon_te, X_unmon_te, unmon_label, n_thresholds=1000):
    """
    Đánh giá open-world với nhiều threshold dày hơn.
    Trả về list of dict với:
    TH, TP, TN, FP, FN, Precision, Recall, F1
    """

    # 0 -> 1 với 1000 điểm
    thresholds = np.linspace(0.0, 1.0, n_thresholds)

    # Predict 1 lần
    prob_mon   = model.predict(X_mon_te,   verbose=0)
    prob_unmon = model.predict(X_unmon_te, verbose=0)

    results = []

    header = f"{'TH':>7} {'TP':>6} {'TN':>6} {'FP':>6} {'FN':>6} {'Prec':>8} {'Rec':>8} {'F1':>8}"
    print(header)
    print("-" * len(header))

    for idx, TH in enumerate(thresholds):
        TP = FP = TN = FN = 0

        # ---------------- monitored ----------------
        for probs in prob_mon:
            best_class = np.argmax(probs)
            best_prob  = probs[best_class]

            if best_class != unmon_label and best_prob >= TH:
                TP += 1
            else:
                FN += 1

        # ---------------- unmonitored ----------------
        for probs in prob_unmon:
            best_class = np.argmax(probs)
            best_prob  = probs[best_class]

            if best_class != unmon_label and best_prob >= TH:
                FP += 1
            else:
                TN += 1

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)
                    if (precision + recall) > 0 else 0.0)

        row = {
            "TH": TH,
            "TP": TP,
            "TN": TN,
            "FP": FP,
            "FN": FN,
            "Precision": precision,
            "Recall": recall,
            "F1": f1
        }
        results.append(row)

        # log thưa thôi cho đỡ spam terminal
        if idx % 100 == 0 or idx == len(thresholds)-1:
            print(f"{TH:7.4f} {TP:6d} {TN:6d} {FP:6d} {FN:6d} "
                  f"{precision:8.4f} {recall:8.4f} {f1:8.4f}")

    return results

 
def run_open_world():
    print("=" * 60)
    print("OPEN-WORLD EVALUATION")
    print("=" * 60)
 
    # ── 1. Load monitored (closed) data ──────────────────────────
    print("\n[1/4] Loading monitored data (closed-world)...")
    X_mon, y_mon = load_dataset(CLOSE_DATA_DIR)
    num_mon_classes = len(np.unique(y_mon))
    print(f"  Monitored samples: {len(y_mon)}, Classes: {num_mon_classes}")
 
    # ── 2. Load unmonitored (open) data ──────────────────────────
    print("\n[2/4] Loading unmonitored data (open-world)...")
    # Tất cả đều label = unmon_label
    unmon_label = num_mon_classes          # index ngay sau class cuối của monitored
    X_unmon_all, _ = load_dataset(OPEN_DATA_DIR, label_override=unmon_label)
    print(f"  Unmonitored samples available: {len(X_unmon_all)}")
 
    total_unmon_needed = UNMON_TRAIN_SIZE + UNMON_TEST_SIZE
    if len(X_unmon_all) < total_unmon_needed:
        raise ValueError(
            f"Không đủ unmonitored samples! "
            f"Cần {total_unmon_needed}, có {len(X_unmon_all)}."
        )
    rng = np.random.RandomState(42)
    perm = rng.permutation(len(X_unmon_all))
    X_unmon_tr = X_unmon_all[perm[:UNMON_TRAIN_SIZE]]
    X_unmon_te = X_unmon_all[perm[UNMON_TRAIN_SIZE:UNMON_TRAIN_SIZE + UNMON_TEST_SIZE]]
    y_unmon_tr = np.full(UNMON_TRAIN_SIZE, unmon_label, dtype=np.int32)
 
    # ── 3. Split monitored data thành train / test ────────────────
    print("\n[3/4] Splitting monitored data & building training set...")
    X_mon_tr, y_mon_tr, _, _, X_mon_te, _ = split_by_label(X_mon, y_mon)
 
    # Gộp monitored train + unmonitored train
    X_train = np.concatenate([X_mon_tr, X_unmon_tr], axis=0)
    y_train = np.concatenate([y_mon_tr, y_unmon_tr], axis=0)
    X_train, y_train = sk_shuffle(X_train, y_train, random_state=42)
 
    num_classes_ow = unmon_label + 1    # monitored classes + 1 unknown class
    seq_length     = X_train.shape[1]
 
    X_train_c  = X_train[:, :, np.newaxis]
    X_mon_te_c = X_mon_te[:, :, np.newaxis]
    X_unmon_te_c = X_unmon_te[:, :, np.newaxis]
 
    Y_train = to_categorical(y_train, num_classes_ow)
 
    print(f"  Train: {len(y_train)} ({len(y_mon_tr)} mon + {UNMON_TRAIN_SIZE} unmon)")
    print(f"  Test (mon): {len(X_mon_te)} | Test (unmon): {len(X_unmon_te)}")
 
    # ── 4. Train & evaluate ───────────────────────────────────────
    print(f"\n[4/4] Training DFNet ({num_classes_ow} classes, seq_len={seq_length})...")
    model = build_model(num_classes_ow, seq_length)
 
    early_stop = EarlyStopping(monitor='loss', patience=5,
                               restore_best_weights=True, verbose=1)
    model.fit(X_train_c, Y_train,
              batch_size=BATCH_SIZE,
              epochs=EPOCHS_OW,
              callbacks=[early_stop],
              verbose=2)
 
    print("\n── Open-World Precision / Recall @ Thresholds ──")
    # gọi như cũ
    results = ow_evaluation(
        model,
        X_mon_te_c,
        X_unmon_te_c,
        unmon_label,
        n_thresholds=1000   # trước là 15
    )
 
    # ── Save Precision-Recall CSV ─────────────────────────────
    
    df_pr = pd.DataFrame({
        "Threshold": [r["TH"] for r in results],
        "Precision": [r["Precision"] for r in results],
        "Recall":    [r["Recall"] for r in results],
        "F1":        [r["F1"] for r in results],
    })
    
    df_pr.to_csv("tiktok_precision_recall_curve.csv", index=False)
    print("\nSaved: tiktok_precision_recall_curve.csv")
    
    
    # ── Save ROC CSV ──────────────────────────────────────────
    
    total_mon   = len(X_mon_te)
    total_unmon = len(X_unmon_te)
    
    df_roc = pd.DataFrame({
        "Threshold": [r["TH"] for r in results],
        "TPR": [r["TP"] / total_mon if total_mon > 0 else 0 for r in results],
        "FPR": [r["FP"] / total_unmon if total_unmon > 0 else 0 for r in results],
    })
    
    df_roc.to_csv("tiktok_roc_curve.csv", index=False)
    print("Saved: tiktok_roc_curve.csv")
 
    return model, results
 
 

In [ ]:
# =============================================================================
# MAIN
# =============================================================================
 
if __name__ == "__main__":
    # ── Closed-World ──
    cw_model, cw_acc = run_closed_world()
 
    print("\n")
 
    # ── Open-World ──
    ow_model, ow_results = run_open_world()
 